In [1]:
!pip install altair
# pip install "altair<6"



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# I\. Setup and normalization

In [2]:
# Imports
import pandas as pd
import numpy as np
import sys
import altair as alt
from datetime import datetime
from scipy.stats import spearmanr, ttest_ind
import statsmodels.api as sm

os.chdir("/work") # needed for the read_csvs to find files

# make sure /work is first on the import path (ahead of site-packages)
if "/work" not in sys.path:
    sys.path.insert(0, "/work")

pd.set_option('display.max_columns', None)
pd.set_option("display.width", 200)

In [3]:
import altair as alt
# alt.renderers.enable("mimetype")
alt.renderers.enable("html")
alt.data_transformers.disable_max_rows()

from utils.viz_theme import enable
enable()

# See what themes exist and what's active
print("Themes available:", list(alt.themes.names()))
print("Active theme:", alt.themes.active)


Themes available: ['carbong10', 'carbong100', 'carbong90', 'carbonwhite', 'dark', 'default', 'excel', 'fivethirtyeight', 'ggplot2', 'googlecharts', 'latimes', 'none', 'opaque', 'powerbi', 'quartz', 'team_theme', 'urbaninstitute', 'vox']
Active theme: team_theme


In [4]:
# Load the key dataframe

albums_df = pd.read_csv("./pipeline/4.7.Albums_analytics_set.csv")
wide_df = pd.read_csv("./pipeline/4.7.Wide_analytics_set.csv")

print(albums_df.columns.tolist())
print(wide_df.columns.tolist())

['tmdb_id', 'film_title', 'film_adult', 'film_runtime_min', 'film_genres', 'film_rating', 'film_vote_count', 'film_mpaa_rating', 'film_original_title', 'film_language_name', 'film_imdb_id', 'film_wikidata_id', 'film_countries', 'film_year', 'film_release_date', 'film_popularity', 'film_budget', 'film_revenue', 'film_studios', 'film_director', 'film_soundtrack_composer_raw', 'film_top_cast', 'film_keywords', 'film_ingested_at', 'match_method', 'soundtrack_type', 'notes', 'matched_at', 'release_group_id', 'release_group_mbid', 'album_title', 'rg_primary_type', 'rg_secondary_types', 'release_id', 'release_mbid', 'release_title', 'release_status', 'release_packaging', 'barcode', 'release_language', 'release_script', 'release_comment', 'album_us_release_date', 'album_us_release_year', 'album_us_release_month_min_observed', 'album_us_release_day_min_observed', 'us_date_has_missing_month', 'us_date_has_missing_day', 'us_any_event_missing_month', 'us_any_event_missing_day', 'rg_tags_text', 're

In [5]:
display(albums_df.sample(5))

,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,film_imdb_id,film_wikidata_id,film_countries,film_year,film_release_date,film_popularity,film_budget,film_revenue,film_studios,film_director,film_soundtrack_composer_raw,film_top_cast,film_keywords,film_ingested_at,match_method,soundtrack_type,notes,matched_at,release_group_id,release_group_mbid,album_title,rg_primary_type,rg_secondary_types,release_id,release_mbid,release_title,release_status,release_packaging,barcode,release_language,release_script,release_comment,album_us_release_date,album_us_release_year,album_us_release_month_min_observed,album_us_release_day_min_observed,us_date_has_missing_month,us_date_has_missing_day,us_any_event_missing_month,us_any_event_missing_day,rg_tags_text,release_tags_text,label_names,label_mbids,label_tags_text,rg_rating,rg_rating_count,release_meta_date_added,release_meta_info_url,release_meta_amazon_asin,release_meta_cover_art_presence,release_status_clean,keep_row,is_canonical_soundtrack,is_canonical_songtrack,canonical_rule,canonical_songtrack_rule,days_since_film_release,days_since_album_release,ambient_experimental,classical_orchestral,electronic,hip_hop_rnb,pop,rock,world_folk,film_is_action,film_is_adventure,film_is_animation,film_is_comedy,film_is_crime,film_is_documentary,film_is_drama,film_is_family,film_is_fantasy,film_is_history,film_is_horror,film_is_music,film_is_mystery,film_is_romance,film_is_science_fiction,film_is_tv_movie,film_is_thriller,film_is_war,film_is_western,vote_count_above_500,composer_primary_clean,award_year,oscar_score_nominee,oscar_score_winner,oscar_song_nominee,oscar_song_winner,globes_score_nominee,globes_score_winner,globes_song_nominee,globes_song_winner,critics_score_nominee,critics_score_winner,critics_song_nominee,critics_song_winner,bafta_score_nominee,bafta_score_winner,lfm_album_status,lfm_album_listeners,lfm_album_playcount,lfm_album_url,lfm_album_query_method,lfm_album_pulled_at,log_lfm_album_listeners,log_lfm_album_playcount
58,290762,Miss You Already,False,112,"Comedy, Drama",7.383,682,PG-13,Miss You Already,English,tt2245003,Q18152748,United Kingdom,2015,2015-09-12,1.2889,0,8200000,"New Sparta Films, S Films",Catherine Hardwicke,Harry Gregson-Williams,"Drew Barrymore, Toni Collette, Dominic Cooper,...","friendship, pregnancy, female friendship, love...",2026-01-19 01:36:05.016 -0500,imdb_exact,songs,NaN,2026-01-25 02:45:25.371 -0500,2633419,9eee5926-dce1-4851-9e9e-df77e20887e3,Miss You Already: Original Motion Picture Soun...,Album,Soundtrack,3016083.0,b184ad0d-a270-4531-be7d-30f4cb1288a3,Miss You Already: Original Motion Picture Soun...,Official,NaN,8.887515e+11,English,Latin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,electronic | indie rock | pop | rock,NaN,Sony Classical | Sony Music,10920823-9ed8-45d9-adb3-69fc22475ab0 | 9e6b4d7...,classical | big three | clean up | pop rock | ...,NaN,NaN,2021-03-14 00:46:21.028 -0500,NaN,NaN,absent,Official,True,True,False,single_album_film,NaN,3794,NaN,False,False,True,False,True,True,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,Harry Gregson-Williams,0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,ok,1.0,1.0,https://www.last.fm/music/Various+Artists/Miss...,text,2026-01-28 21:57:47.610894+00:00,0.000000,0.000000
988,399035,The Commuter,False,104,"Action, Thriller, Mystery",6.400,4549,PG-13,The Commuter,English,tt1590193,Q27566067,United States of America / France / Spain,2018,2018-01-11,2.6807,30000000,119942387,"StudioCanal, TF1 Films Production, The Picture...",Jaume Collet-Serra,Roque Baños,"Liam Neeson, Patrick Wilson, Sam Neill, Jonath...","conspiracy, train",2026-01-19 01:36:05.812 -0500,imdb_exact,songs,NaN,2026-01-25 02:45:25.371 -0500,1911979,311578c6-d445-492b-b27c-e63ff9cfa638,The Commuter (Original Motion Picture Soundtrack),Album,Soundtrack

In [6]:
nullcount = albums_df.isna().sum()
print(nullcount.index.tolist())
print(nullcount.tolist())

['tmdb_id', 'film_title', 'film_adult', 'film_runtime_min', 'film_genres', 'film_rating', 'film_vote_count', 'film_mpaa_rating', 'film_original_title', 'film_language_name', 'film_imdb_id', 'film_wikidata_id', 'film_countries', 'film_year', 'film_release_date', 'film_popularity', 'film_budget', 'film_revenue', 'film_studios', 'film_director', 'film_soundtrack_composer_raw', 'film_top_cast', 'film_keywords', 'film_ingested_at', 'match_method', 'soundtrack_type', 'notes', 'matched_at', 'release_group_id', 'release_group_mbid', 'album_title', 'rg_primary_type', 'rg_secondary_types', 'release_id', 'release_mbid', 'release_title', 'release_status', 'release_packaging', 'barcode', 'release_language', 'release_script', 'release_comment', 'album_us_release_date', 'album_us_release_year', 'album_us_release_month_min_observed', 'album_us_release_day_min_observed', 'us_date_has_missing_month', 'us_date_has_missing_day', 'us_any_event_missing_month', 'us_any_event_missing_day', 'rg_tags_text', 're

# II\. Correlation analysis

### II\.1 Setup analytics frame

Let's add a few derived features into albums\_df\. First we'll add n\_tracks which counts the number of tracks in an album\. This will be a new derived feature at the album level\.

In [7]:

# Add n_tracks: number of tracks

# Count unique tracks per (release_group_mbid, tmdb_id)
# If you have already computed n_tracks before, this will overwrite it deterministically.
n_tracks_df = (
    wide_df[["release_group_mbid", "tmdb_id", "track_id"]]
    .groupby(["release_group_mbid", "tmdb_id"], as_index=False)
    .agg(n_tracks=("track_id", "nunique"))
)

# Drop existing column to avoid _x/_y suffixes on repeated runs
if "n_tracks" in albums_df.columns:
    albums_df = albums_df.drop(columns=["n_tracks"])

albums_df = albums_df.merge(
    n_tracks_df,
    on=["release_group_mbid", "tmdb_id"],
    how="left",
    validate="1:1",
)

print("n_tracks added. Nulls:", int(albums_df["n_tracks"].isna().sum()))
print(albums_df["n_tracks"].describe())

n_tracks added. Nulls: 8
count    1543.000000
mean       20.262476
std         8.492489
min         1.000000
25%        15.000000
50%        20.000000
75%        25.000000
max        87.000000
Name: n_tracks, dtype: float64


These award features were derived after inspecting the correlation matrix for the original award indicators\. The analysis showed that Oscar, Golden Globe, and Critics Choice variables were correlated with each other and largely reflect the same “U\.S\. major award recognition” signal, while BAFTA behaved more independently\. Based on that, I collapsed the U\.S\. awards into combined features but kept score and song separate\. For modeling, I use nominee counts only \(rather than also including winner counts\) because winners are essentially a subset of nominees and including both in the same regression would be redundant and can make coefficients unstable\. BAFTA is left as simple nominee/winner flags since it represents a single ceremony and doesn’t benefit from aggregation\.

In [8]:
# ------------------------------------------------------------
# Derived award attributes (Option 2: nominees only, everywhere)
# ------------------------------------------------------------

# US-major awards: keep score vs song separate (nominees only)
us_score_nominee_cols = [
    "oscar_score_nominee",
    "globes_score_nominee",
    "critics_score_nominee",
]

us_song_nominee_cols = [
    "oscar_song_nominee",
    "globes_song_nominee",
    "critics_song_nominee",
]

# BAFTA (nominee flag only)
bafta_nominee_col = ["bafta_score_nominee"]

# Derived columns to create
derived_award_cols = [
    "us_score_nominee_count",
    "us_song_nominee_count",
    "bafta_nominee",
]

# Drop derived columns if they already exist
existing = [c for c in derived_award_cols if c in albums_df.columns]
if existing:
    albums_df = albums_df.drop(columns=existing)

# Build US-major nominee counts
albums_df["us_score_nominee_count"] = albums_df[us_score_nominee_cols].sum(axis=1).astype(int)
albums_df["us_song_nominee_count"]  = albums_df[us_song_nominee_cols].sum(axis=1).astype(int)

# Build BAFTA nominee flag
albums_df["bafta_nominee"] = (albums_df[bafta_nominee_col].sum(axis=1) > 0).astype(int)

print("Derived award cols added. Sums:")
print(albums_df[derived_award_cols].sum().sort_values(ascending=False))


Derived award cols added. Sums:
us_score_nominee_count    152
us_song_nominee_count     132
bafta_nominee              50
dtype: int64


This step adds a simple “composer experience” feature\. It counts how many soundtrack albums each composer appears on in our dataset \(composer\_album\_count\) and merges that value back onto every album row for use in later analysis and modeling\.

In [9]:
# Add a composer_album_count feature to each album representing the number of albums
# this composer is associated with

composer_counts = (
    albums_df[["composer_primary_clean", "release_group_mbid", "tmdb_id"]]
    .groupby("composer_primary_clean", as_index=False)
    .agg(composer_album_count=("release_group_mbid", "count"))
)

if "composer_album_count" in albums_df.columns:
    albums_df = albums_df.drop(columns=["composer_album_count"])

albums_df = albums_df.merge(
    composer_counts,
    on="composer_primary_clean",
    how="left",
    validate="m:1",
)

print("composer_album_count added. Nulls:", int(albums_df["composer_album_count"].isna().sum()))
print(albums_df["composer_album_count"].describe())

composer_album_count added. Nulls: 0
count    1551.000000
mean        9.188266
std         8.431930
min         1.000000
25%         2.000000
50%         6.000000
75%        16.000000
max        30.000000
Name: composer_album_count, dtype: float64


Album genre flags currently are stored as objects with values True, False or NaN \(when no tags were present\)\. For the correlation analysis, we should convert these to False

In [10]:

genre_cols = [
    "ambient_experimental", "classical_orchestral", "electronic",
    "hip_hop_rnb", "pop", "rock", "world_folk"
]

# Idempotent: bool/float/object -> int(0/1); NaN -> 0
albums_df[genre_cols] = (
    albums_df[genre_cols]
    .fillna(False)     # NaN -> False
    .astype(bool)      # anything truthy -> True, False stays False
    .astype(int)       # True/False -> 1/0
)

This defines the final set of film\- and album\-level predictors \(plus derived award features\) and builds album\_analytics\_df, the modeling\-ready dataframe we use for diagnostics and regression with log\_lfm\_album\_listeners as the target\.

In [11]:

film_ids = ["tmdb_id"]

film_features = [
    # --- Exposure / reach ---
    "film_vote_count", "film_popularity",

    # --- Economics (optional in models, but retained) ---
    "film_budget", "film_revenue",

    # --- Quality ---
    "film_rating",

    # --- Timing and Structure ---
    "days_since_film_release","film_runtime_min",

    # --- Film genre indicators ---
    "film_is_action", "film_is_adventure", "film_is_animation", "film_is_comedy", "film_is_crime", 
    "film_is_documentary", "film_is_drama", "film_is_family", "film_is_fantasy", "film_is_history",
    "film_is_horror", "film_is_music", "film_is_mystery", "film_is_romance", "film_is_science_fiction", 
    "film_is_tv_movie", "film_is_thriller", "film_is_war", "film_is_western"
]

album_ids = [
    "release_group_mbid",
    "tmdb_id",
]

album_features = [
    # --- Core ---
    "days_since_album_release", "n_tracks", "composer_album_count",

    # --- Album genre indicators (0/1) ---
    "ambient_experimental", "classical_orchestral", "electronic", "hip_hop_rnb", "pop", "rock", "world_folk",
]

y_feature = ['log_lfm_album_listeners']

album_analytics_df = albums_df[film_features + album_features + derived_award_cols + y_feature]
# album_analytics_df = albums_df[film_features + album_features + award_cols + y_feature]

### II\.2 Correlation table

This section explores pairwise relationships across film, album, genre, award, timing, and composer features using Spearman rank correlations\. Spearman is used instead of Pearson because many variables in this dataset are binary indicators or heavy\-tailed measures \(e\.g\., film vote counts, budgets, and revenues\), and relationships are often monotonic rather than strictly linear\. Rank\-based correlations provide a stable view of association structure in this setting without assuming linearity or allowing extreme values to dominate\.

In [12]:
corr_df = album_analytics_df.select_dtypes(include=["number", "bool"])
print(corr_df.shape, album_analytics_df.shape)   # Verify that we don't lose columns

corr_spearman = corr_df.corr(method="spearman")

corr_spearman

(1551, 40) (1551, 40)


,film_vote_count,film_popularity,film_budget,film_revenue,film_rating,days_since_film_release,film_runtime_min,film_is_action,film_is_adventure,film_is_animation,film_is_comedy,film_is_crime,film_is_documentary,film_is_drama,film_is_family,film_is_fantasy,film_is_history,film_is_horror,film_is_music,film_is_mystery,film_is_romance,film_is_science_fiction,film_is_tv_movie,film_is_thriller,film_is_war,film_is_western,days_since_album_release,n_tracks,composer_album_count,ambient_experimental,classical_orchestral,electronic,hip_hop_rnb,pop,rock,world_folk,us_score_nominee_count,us_song_nominee_count,bafta_nominee,log_lfm_album_listeners
film_vote_count,1.000000,0.595819,0.602685,0.665682,0.262411,0.172205,0.287052,0.246509,0.280568,0.030921,-0.017559,0.002313,-0.095996,-0.174027,0.024991,0.068928,-0.047104,-0.004302,-0.023196,-0.011546,-0.089806,0.225143,-0.047468,-0.026963,-0.024075,-0.047346,0.191572,0.156944,0.297717,0.051313,0.234627,0.061157,0.053874,0.141297,0.111970,0.056291,0.215460,0.101418,0.204830,0.273417
film_popularity,0.595819,1.000000,0.542423,0.567476,0.293854,-0.307606,0.275878,0.296820,0.301162,0.146176,-0.009100,-0.000528,-0.134276,-0.302232,0.103177,0.122516,-0.123436,0.012838,-0.035747,-0.039291,-0.076416,0.209597,-0.045824,-0.008213,-0.033598,-0.033962,-0.110168,0.188625,0.225876,0.039397,0.111822,0.007522,0.018160,0.032592,0.045494,0.040994,0.144851,0.093271,0.185128,0.144696
film_budget,0.602685,0.542423,1.000000,0.732502,0.062519,-0.043251,0.329971,0.341802,0.428009,0.119901,0.062652,0.010523,-0.121066,-0.253613,0.174708,0.134141,-0.027152,-0.171004,-0.015488,-0.105260,-0.133289,0.232920,-0.082052,-0.111680,-0.026399,-0.021621,0.028562,0.241227,0.365630,-0.023027,0.205341,-0.028657,0.024446,0.080140,0.040091,0.012018,0.134039,0.124239,0.178694,0.185791
film_revenue,0.665682,0.567476,0.732502,1.000000,0.152084,0.083276,0.212553,0.232482,0.338693,0.145557,0.055451,-0.044375,-0.090629,-0.231142,0.176855,0.085681,-0.041705,-0.038332,0.008462,-0.070584,-0.081537,0.161403,-0.097448,-0.114545,-0.055363,-0.060521,0.181470,0.214802,0.295225,-0.003022,0.241266,0.005840,0.043793,0.124160,0.071430,0.034246,0.141437,0.099579,0.180028,0.187874
film_rating,0.262411,0.293854,0.062519,0.152084,1.000000,-0.134994,0.299487,-0.100180,0.071921,0.225735,0.007111,-0.061881,0.095539,0.241113,0.161709,-0.021275,0.143303,-0.283938,0.107171,-0.133279,0.092900,-0.077671,0.062142,-0.243078,0.101087,0.028450,-0.163755,0.064600,0.025425,0.043010,0.050512,0.011678,0.044835,0.069758,0.018812,0.092632,0.212942,0.136781,0.188146,0.146537
days_since_film_release,0.172205,-0.307606,-0.043251,0.083276,-0.134994,1.000000,-0.100049,-0.017308,0.001049,-0.049261,-0.013581,0.012245,0.051714,0.147871,-0.029492,-0.070279,0.043695,-0.076704,0.023122,-0.037835,0.024957,-0.043521,0.055825,-0.024431,0.041538,0.029799,0.988909,-0.119422,0.005538,0.065142,0.181516,0.123746,0.102052,0.111566,0.107826,0.013746,-0.050661,-0.036114,-0.036973,0.177155
film_runtime_min,0.287052,0.275878,0.329971,0.212553,0.299487,-0.100049,1.000000,0.141841,0.041647,-0.296796,-0.214558,0.075578,-0.024306,0.217619,-0.260432,-0.075856,0.204090,-0.250364,0.019746,-0.054564,0.039054,0.072672,-0.019242,-0.045349,0.064947,0.064922,-0.105827,0.077320,0.170523,0.035477,0.121666,0.011346,0.043294,0.041882,0.070128,0.055241,0.214585,0.126509,0.186907,0.136505
film_is_action,0.246509,0.296820,0.341802,0.232482,-0.100180,-0.017308,0.141841,1.000000,0.299132,-0.067740,-0.130810,0.117347,-0.058042,-0.348919,-0.133957,0.029480,-0.091936,-0.162191,-0.105861,-0.145572,-0.203049,0.274087,-0.042508,0.117966,0.020620,-0.035437,0.013835,0.074461,0.127390,0.023791,0.093754,0.055085,0.011896,-0.037474,0.026133,-0.031352,-0.058930,-0.056754,0.030333,0.036933
film_is_adventure,0.280568,0.301162,0.428009,0.338693,0.071921,0.001049,0.041647,0.299132,1.000000,0.338495,0.065214,-0.159400,-0.035824,-0.325207,0.359497,0.272385,-0.108516,-0.222905,-0.042664,-0.131148,-0.159870,0.232196,0.028865,-0.203414,-0.

In [13]:
def corr_to_long(corr_df: pd.DataFrame) -> pd.DataFrame:
    return (
        corr_df
        .reset_index()
        .melt(id_vars="index", var_name="var_y", value_name="corr")
        .rename(columns={"index": "var_x"})
    )


In [14]:
corr_long = corr_to_long(corr_spearman)   # or corr_pearson

heatmap = (
    alt.Chart(corr_long)
    .mark_rect()
    .encode(
        x=alt.X(
            "var_x:O",
            title=None,
            sort=None,
            axis=alt.Axis(labelAngle=-45, labelFontSize=13, titleFontSize=13)
        ),
        y=alt.Y(
            "var_y:O",
            title=None,
            sort=None,
            axis=alt.Axis(labelFontSize=13, titleFontSize=13)
        ),
        color=alt.Color(
            "corr:Q",
            scale=alt.Scale(
                range=["#CC0000", "#F3F0E6", "#1195B2"],
                domain=[-1, 1],
                domainMid=0,
                clamp=True
            ),
            title="Correlation"
        ),
        tooltip=[
            alt.Tooltip("var_x:N", title="X"),
            alt.Tooltip("var_y:N", title="Y"),
            alt.Tooltip("corr:Q", format=".2f", title="ρ"),
        ],
    )
    .properties(
        width=700,
        height=700,
        autosize=alt.AutoSizeParams(type="pad"),
        title="Correlation Heatmap (Spearman)"
    )
)

heatmap

alt.Chart(...)

In [15]:
heatmap.to_dict()["height"]
heatmap.to_dict().get("height"), heatmap.to_dict().get("autosize")


(700, {'type': 'pad'})

### II\.3 Listener\-focused correlation view \(Lollipop Plot\)

In the previous section, we used a Spearman correlation heatmap to understand how features relate to one another and to surface clusters of highly collinear variables\. That view is helpful for diagnosing redundancy, but it’s less useful when the goal is to understand what actually moves with album popularity itself\. Here, we narrow the focus to a single variable of interest—log\-transformed album listener counts—and ask a simpler question: which features tend to scale up or down as album popularity increases\. Because this is a question about relative magnitude rather than rank, we use Pearson correlations on log\-transformed values, which lets us compare strength and direction of association in a more interpretable way\.

Question:  Which album\- and film\-level features tend to scale with soundtrack album popularity?

In [16]:
target = corr_df.columns[-1]

corr_sorted = (
    corr_df
    .corr(method="pearson")[target]   # correlation with target
    .drop(target)                     # remove self-correlation
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

corr_sorted

film_vote_count             0.223075
film_revenue                0.182448
film_budget                 0.158100
film_rating                 0.145917
days_since_film_release     0.144104
film_runtime_min            0.140124
pop                         0.133150
rock                        0.132747
classical_orchestral        0.129576
days_since_album_release    0.119192
bafta_nominee               0.102702
electronic                  0.091477
us_score_nominee_count      0.088720
film_is_adventure           0.076121
ambient_experimental        0.074213
world_folk                  0.061290
film_is_science_fiction     0.058790
film_is_music               0.057859
film_is_horror             -0.056619
film_is_thriller           -0.053758
us_song_nominee_count       0.050524
hip_hop_rnb                 0.048809
film_is_comedy             -0.046221
film_popularity             0.044599
film_is_romance             0.040916
film_is_fantasy             0.038316
film_is_tv_movie           -0.035334
f

To get an initial read on which variables move with soundtrack popularity, this lollipop chart ranks features by their Pearson correlation with log\(album listeners\)\. Each row shows one feature’s correlation value: the dot marks the magnitude and direction, while the “stick” connects it back to zero to make weak vs\. strong relationships easy to compare at a glance\. A dashed vertical line at 0 highlights features with essentially no linear association; points to the right indicate positive relationships and points to the left indicate negative ones \(with color reinforcing the direction\)\. This is an exploratory, pairwise view—useful for spotting broad patterns and potential predictors—but it does not control for overlap between features the way the regression model does\.

In [17]:
# Convert series to a dataframe
corr_df_plot = (
    corr_sorted
    .rename("corr")
    .reset_index()
    .rename(columns={"index": "feature"})
)

corr_df_plot["abs_corr"] = corr_df_plot["corr"].abs()
corr_df_plot["zero"] = 0

x_domain = [
    float(corr_df_plot["corr"].min()) - 0.02,
    float(corr_df_plot["corr"].max()) + 0.02
]

x_axis = alt.X(
    "corr:Q",
    title="Correlation with log(album listeners)",
    scale=alt.Scale(domain=x_domain)
)

y_axis = alt.Y(
    "feature:N",
    sort=alt.SortField("abs_corr", order="descending"),
    title=None
)

# Draw a lollipop chart of correlations
sticks = alt.Chart(corr_df_plot).mark_rule(strokeWidth=2).encode(
    y=y_axis,
    x=alt.X("zero:Q", scale=alt.Scale(domain=x_domain), title="Correlation with log(album listeners)"),
    x2="corr:Q",
    color=alt.condition(
        "datum.corr >= 0",
        alt.value("#1195B2"),
        alt.value("#CC0000")
    )
)

dots = alt.Chart(corr_df_plot).mark_circle(size=80).encode(
    y=y_axis,
    x=x_axis,
    color=alt.condition(
        "datum.corr >= 0",
        alt.value("#1195B2"),
        alt.value("#CC0000")
    ),
    tooltip=[
        alt.Tooltip("feature:N", title="Feature"),
        alt.Tooltip("corr:Q", title="Correlation", format=".3f")
    ]
)

zero_line = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(
    strokeDash=[4, 4], opacity=0.6
).encode(
    x=alt.X("x:Q", scale=alt.Scale(domain=x_domain))
)

chart = (sticks + dots + zero_line).properties(
    width=650,
    height=900,
    title={
        "text": "Lollipop Plot: Which features scale with album popularity?",
        "subtitle": "Pearson correlations with log(album listeners); no single feature dominates"
    }
)
chart

alt.LayerChart(...)

Findings: No single feature stands out as a strong driver of soundtrack album popularity\. The strongest correlations are modest and are led by broad measures of film visibility—such as vote counts, revenue, budget, and ratings—rather than soundtrack\-specific attributes or awards\. Genre indicators and nomination flags show weaker, mixed relationships, and many film\-level features cluster close to zero\. Even high\-profile awards \(including major nominations and wins\) exhibit only small positive correlations\. Taken together, the pattern suggests that album popularity reflects the combined effect of many small signals tied to overall film exposure, rather than any one dominant creative, genre, or awards\-related factor\.

### II\.4 Correlation sanity check

Before moving on, we run a small set of correlation sanity checks to ensure that the relationships observed so far are not artifacts of a particular correlation metric or driven by a small number of extreme albums\. These checks are not intended to uncover new patterns, but to confirm that the feature–popularity relationships are stable under reasonable alternative assumptions\.

Question:  Do we tell the same "top correlated features" story if we compute correlations using Pearson \(scale\-based\) vs Spearman \(rank\-based\)?

In [18]:
# ------------------------------------------------------------
# Correlation sanity check (target-only):
# Do we get a similar "top features" story under Pearson vs Spearman?
#
# Why this matters:
# - Spearman (rank-based) is robust to heavy tails and nonlinearity.
# - Pearson (scale-based) reflects magnitude relationships (especially in log space).
# If the top features and overall rankings are similar under both, our conclusions
# are less sensitive to the choice of correlation metric.
# ------------------------------------------------------------

target = corr_df.columns[-1]
TOP_N = 15

# Keep only numeric columns and drop rows with missing values so comparisons are apples-to-apples
df = corr_df.dropna()

# Correlations with the target (one number per feature)
pearson = df.corr(method="pearson")[target].drop(target)
spearman = df.corr(method="spearman")[target].drop(target)

# Top-N overlap (by absolute correlation)
top_pearson = set(pearson.abs().nlargest(TOP_N).index)
top_spearman = set(spearman.abs().nlargest(TOP_N).index)
overlap = sorted(top_pearson & top_spearman)

# Overall agreement: Spearman correlation between the rank orders of |corr|
# This feels very meta, but run Spearman correlation on the top ranking features
# that came out of the original pearson and original spearman correlation
rank_agreement = (
    pearson.abs().rank(ascending=False)
    .corr(spearman.abs().rank(ascending=False), method="spearman")
)

# Display: one compact summary table + two headline stats
summary = pd.DataFrame({
    "pearson": pearson,
    "spearman": spearman,
})
summary["abs_pearson"] = summary["pearson"].abs()
summary["abs_spearman"] = summary["spearman"].abs()

print(f"Top-{TOP_N} overlap: {len(overlap)} / {TOP_N}")
print(f"Rank agreement (ρ on ranks of |corr|): {rank_agreement:.3f}")
print("Overlapping features:", overlap)

display(
    summary.sort_values("abs_pearson", ascending=False)
           .head(TOP_N)[["pearson", "spearman"]]
           .round(3)
)


Top-15 overlap: 15 / 15
Rank agreement (ρ on ranks of |corr|): 0.951
Overlapping features: ['bafta_nominee', 'classical_orchestral', 'days_since_album_release', 'days_since_film_release', 'electronic', 'film_budget', 'film_is_music', 'film_popularity', 'film_rating', 'film_revenue', 'film_runtime_min', 'film_vote_count', 'pop', 'rock', 'us_score_nominee_count']


,pearson,spearman
film_vote_count,0.248,0.315
film_revenue,0.223,0.246
film_popularity,0.193,0.229
film_budget,0.180,0.233
film_runtime_min,0.166,0.188
classical_orchestral,0.155,0.179
rock,0.147,0.133
film_rating,0.133,0.136
pop,0.132,0.144
electronic,0.120,0.141


Findings: The features most closely associated with album popularity are highly consistent under both Pearson and Spearman correlations\. With the inclusion of film runtime, the top\-15 features now align perfectly across the two methods \(15/15 overlap\), and the overall agreement in feature rankings remains very high \(ρ ≈ 0\.88\)\. This indicates that the results are not sensitive to whether correlation is measured on the original scale or by rank ordering\.

Question: Are our correlations with album popularity being driven by a tiny number of blockbuster albums?

In [19]:
# ============================================================
# Test 2 — Trim extremes (outlier sensitivity)
# ============================================================

#
# Output:
# - stability of the correlation vector before vs after trimming (one number)
#   for Pearson and Spearman
# ------------------------------------------------------------

TRIM_Q = 0.99  # drop top 1% of albums by target

thr = df[target].quantile(TRIM_Q)
df_trim = df[df[target] <= thr]

pearson_trim = df_trim.corr(method="pearson")[target].drop(target)
spearman_trim = df_trim.corr(method="spearman")[target].drop(target)

# "Vector stability": correlation between the full and trimmed correlation vectors
pearson_stability = pearson.corr(pearson_trim, method="pearson")
spearman_stability = spearman.corr(spearman_trim, method="pearson")

print("\nTest 2 — Trim extremes")
print(f"Trim threshold (top {(1-TRIM_Q)*100:.1f}% removed): {thr:.3f}")
print(f"Rows kept: {len(df_trim):,} / {len(df):,}")
print(f"Pearson stability (full vs trimmed):  {pearson_stability:.3f}")
print(f"Spearman stability (full vs trimmed): {spearman_stability:.3f}")


Test 2 — Trim extremes
Trim threshold (top 1.0% removed): 11.292
Rows kept: 734 / 742
Pearson stability (full vs trimmed):  0.956
Spearman stability (full vs trimmed): 0.991


Findings: Removing the most extreme albums by listener count has little effect on the overall correlation structure\. Correlations before and after trimming remain highly aligned \(Pearson = 0\.96; Spearman = 0\.99\), indicating that the observed relationships are not driven by a small number of blockbuster soundtracks and are robust to the presence of outliers\.

# III\. Hypothesis testing

### III\.1 Popular films \-\-\> more listened albums

Hypothesis: Albums tied to box office hits have higher listener counts\.

Film popularity can be measured in several closely related ways, including vote counts, budget, box office revenue, and platform\-level popularity scores\. These variables are known to be highly collinear and largely reflect the same underlying concept: overall film exposure\. Rather than treating them as independent predictors, we test each as an alternative operationalization of film popularity to evaluate whether the same directional relationship holds across measures\. This approach allows us to assess the robustness of the “popular film → popular soundtrack” relationship without over\-interpreting any single metric\.

In [20]:
# Variables of interest
# Collinear features
collinear = ['film_vote_count', 'film_budget', 'film_popularity', 'film_revenue']

y = album_analytics_df["log_lfm_album_listeners"]

results = {}
for feature in collinear:
    x = album_analytics_df[feature]

    # Drop missing values pairwise
    mask = x.notna() & y.notna()
    x_valid = x[mask]
    y_valid = y[mask]

    # Spearman rank correlation test
    rho, p_value = spearmanr(x_valid, y_valid)

    alpha = 0.05
    if p_value < alpha:
        test = f"Result: Reject H₀ — evidence of a positive association with {feature}."
    else:
        test = f"Result: Fail to reject H₀ — no evidence of association with {feature}."

    results[feature] = (round(rho, 6), p_value, test)

display(pd.DataFrame(results, index=["rho", "p_value", "test"]).T.round(2))

,rho,p_value,test
film_vote_count,0.273417,0.0,Result: Reject H₀ — evidence of a positive ass...
film_budget,0.185791,0.0,Result: Reject H₀ — evidence of a positive ass...
film_popularity,0.144696,0.0,Result: Reject H₀ — evidence of a positive ass...
film_revenue,0.187874,0.0,Result: Reject H₀ — evidence of a positive ass...


Findings: All four film popularity measures show a statistically significant positive association with album listener counts\. The strength of the relationship varies by metric, with vote count exhibiting the strongest correlation, followed by budget and revenue, and popularity scores somewhat weaker\. Given the high degree of collinearity among these features, these results should be interpreted as consistent evidence of a shared exposure effect rather than four independent findings\. Taken together, they support the conclusion that soundtracks tied to more widely seen and discussed films tend to attract more listeners\.

### III\.2 Prestige factor: do awards matter?

Hypothesis: Award\-recognized scores attract more listeners than non\-recognized scores\.

In [21]:
y = album_analytics_df["log_lfm_album_listeners"]

award_features = ["us_score_nominee_count", "us_song_nominee_count", "bafta_nominee"]

results = {}
alpha = 0.05

for feature in award_features:
    x = album_analytics_df[feature]

    # Pairwise drop missing values
    mask = x.notna() & y.notna()
    x_valid = x[mask]
    y_valid = y[mask]

    # --- Welch setup: define "recognized" vs "not recognized" ---
    # Recognized = x > 0 (works for counts and 0/1 flags)
    y_recognized = y_valid[x_valid > 0]
    y_not        = y_valid[x_valid == 0]

    # Welch’s t-test (unequal variances)
    t_stat, p_value = ttest_ind(
        y_recognized,
        y_not,
        equal_var=False,
        nan_policy="omit"
    )

    # Direction (helpful for interpretation)
    mean_rec = y_recognized.mean()
    mean_not = y_not.mean()
    diff = mean_rec - mean_not  # positive means recognized > not

    if p_value < alpha and diff > 0:
        test = f"Reject H₀ — recognized group has higher mean listeners for {feature}."
    elif p_value < alpha and diff < 0:
        test = f"Reject H₀ — recognized group has LOWER mean listeners for {feature}."
    else:
        test = f"Fail to reject H₀ — no clear mean difference for {feature}."

    results[feature] = (len(y_recognized), len(y_not), mean_rec, mean_not, diff, t_stat, p_value, test)

display(
    pd.DataFrame(
        results,
        index=["n_rec", "n_not", "mean_rec", "mean_not", "mean_diff", "t_stat", "p_value", "test"]
    ).T
    .round(3)
)


,n_rec,n_not,mean_rec,mean_not,mean_diff,t_stat,p_value,test
us_score_nominee_count,77,1474,5.033741,3.850094,1.183647,4.073646,0.000103,Reject H₀ — recognized group has higher mean l...
us_song_nominee_count,63,1488,4.664041,3.876883,0.787159,2.100565,0.039485,Reject H₀ — recognized group has higher mean l...
bafta_nominee,50,1501,5.398273,3.859242,1.53903,3.682956,0.000552,Reject H₀ — recognized group has higher mean l...


Findings: Across all three award indicators, albums associated with award\-recognized scores or songs have higher average listener counts than those without recognition\. Using Welch’s t\-tests to compare mean log listener counts between recognized and non\-recognized groups, the differences are statistically significant for U\.S\. score nominations, U\.S\. song nominations, and BAFTA nominations\. In each case, the award\-recognized group shows meaningfully higher average listeners, supporting the hypothesis that industry recognition is associated with greater soundtrack popularity\.

### III\.3 Genre effects

Hypothesis: Listener interest differs by soundtrack musical style\.

Because we transformed NaNs into False flags for genre in order to build the earlier correlation table, we cannot use album\_analytics\_df for this next hypothesis test\. We need to leverage the original albums\_df \(which still contain NaNs in the rg\_tags\_text column\)\.

In [22]:
genre_features = [
    "ambient_experimental", "classical_orchestral", "electronic",
    "hip_hop_rnb", "pop", "rock", "world_folk"
]

y_col = "log_lfm_album_listeners"

# Only albums with genre tags present
df_genre = albums_df[
    albums_df["rg_tags_text"].notna() &
    albums_df[y_col].notna()
]

alpha = 0.05 / len(genre_features)
results = {}

for g in genre_features:
    # Explicit in-genre vs not-in-genre (both have tags)
    y_in  = df_genre.loc[df_genre[g] == 1, y_col]
    y_out = df_genre.loc[df_genre[g] == 0, y_col]

    t_stat, p_value = ttest_ind(
        y_in, y_out,
        equal_var=False,
        nan_policy="omit"
    )

    mean_in = y_in.mean()
    mean_out = y_out.mean()
    diff = mean_in - mean_out

    test = (
        f"Reject H₀ — mean differs for {g}."
        if p_value < alpha
        else f"Fail to reject H₀ — no clear mean difference for {g}."
    )

    results[g] = (
        len(y_in), len(y_out),
        mean_in, mean_out,
        diff, t_stat, p_value, test
    )

display(
    pd.DataFrame(
        results,
        index=[
            "n_in_genre", "n_not",
            "mean_in", "mean_not",
            "mean_diff",
            "t_stat", "p_value", "test"
        ]
    ).T.round(3)
)


,n_in_genre,n_not,mean_in,mean_not,mean_diff,t_stat,p_value,test
ambient_experimental,54,375,5.09515,4.79548,0.29967,0.733943,0.4655,Fail to reject H₀ — no clear mean difference f...
classical_orchestral,189,240,4.82342,4.840904,-0.017484,-0.066651,0.946891,Fail to reject H₀ — no clear mean difference f...
electronic,87,342,4.889104,4.81898,0.070123,0.209769,0.834175,Fail to reject H₀ — no clear mean difference f...
hip_hop_rnb,31,398,4.813495,4.834736,-0.021241,-0.050201,0.960227,Fail to reject H₀ — no clear mean difference f...
pop,83,346,5.305126,4.719994,0.585132,1.710505,0.089733,Fail to reject H₀ — no clear mean difference f...
rock,57,372,5.585499,4.717929,0.86757,1.825241,0.072477,Fail to reject H₀ — no clear mean difference f...
world_folk,19,410,5.365574,4.80853,0.557044,0.667034,0.512791,Fail to reject H₀ — no clear mean difference f...


Findings: After restricting the analysis to albums with available genre metadata, I tested whether listener interest differed by soundtrack musical style using Welch’s t\-tests\. Across all seven album genres, I fail to reject the null hypothesis\. While some genres \(notably pop and rock\) show higher average listener counts, these differences are not statistically significant once sampling variability is taken into account\. Overall, the results suggest that album\-level musical style alone does not meaningfully differentiate listener popularity in this dataset\. Any apparent differences in means are small and inconsistent, indicating that genre by itself is not a strong driver of soundtrack listening behavior\.

### III\.4 Composer signal

Hypothesis: Established composers have higher\-listened albums, even controlling for film popularity\.

To address this question, I fit a simple linear regression predicting log soundtrack listeners using composer album count as a proxy for composer experience, while controlling for film vote count as a measure of film popularity\. This allows me to test whether composer experience is associated with listener interest beyond the effect of overall film exposure\.

In [23]:
# Select only what we need
cols = [
    "log_lfm_album_listeners",
    "composer_album_count",
    "film_vote_count",
]

df_test = album_analytics_df[cols].dropna()

y = df_test["log_lfm_album_listeners"]
X = df_test[["composer_album_count", "film_vote_count"]]

# Add intercept
X = sm.add_constant(X)

# Fit model
res = sm.OLS(y, X).fit()

print(res.summary())


                               OLS Regression Results                              
Dep. Variable:     log_lfm_album_listeners   R-squared:                       0.052
Model:                                 OLS   Adj. R-squared:                  0.051
Method:                      Least Squares   F-statistic:                     42.54
Date:                     Fri, 13 Feb 2026   Prob (F-statistic):           1.03e-18
Time:                             19:58:27   Log-Likelihood:                -3669.0
No. Observations:                     1551   AIC:                             7344.
Df Residuals:                         1548   BIC:                             7360.
Df Model:                                2                                         
Covariance Type:                 nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------

Findings: After controlling for film popularity, there is no clear evidence that albums by more established composers attract more listeners\. In this model, film vote count shows a strong positive association with soundtrack listeners, which is expected since it reflects overall film exposure\. In contrast, composer album count has a small negative coefficient and a p\-value just above the conventional 0\.05 threshold\. I interpret this as weak and inconclusive evidence that composer experience, as defined here, does not meaningfully increase listener interest once film popularity is taken into account\.

# IV Regression

### IV\.1 Winnow features

Before fitting any regression models, we first narrow the feature set to something interpretable and stable\. The goal here is not aggressive feature selection or optimization, but basic hygiene: grouping predictors by type, reducing obvious noise, and avoiding unnecessary complexity in the first\-pass Ordinary Least Squres \(OLS\)\. We bucket features into continuous and binary predictors, retain all binary indicators \(genres, awards, flags\), and apply a light correlation\-based filter to continuous variables only\. This keeps the regression focused on predictors with at least a minimal linear relationship to log album listeners, while preserving categorical context and avoiding premature assumptions about causal importance\.

In [24]:
# ------------------------------------------------------------
# IV) Regression prep 
# 1) Feature groups (bucket into continuous and binary)
# 2) Light winnowing:
#    - Keep all binary flags (film genres, album genres, awards)
#    - Filter continuous predictors by |Pearson r| >= THRESH vs target
# Light winnowing is intentionally conservative:
# - We only filter continuous predictors
# - Binary flags are retained to preserve categorical context
# - Threshold is low by design; this is not feature selection, just noise reduction
# ------------------------------------------------------------

THRESH = 0.05
target = y_feature[0]  # 'log_lfm_album_listeners'

# -----------------------------
# Step 1) Define feature groups (from existing lists)
# Continuous variables are potential scale drivers in OLS
# Binary variables act as controls / group indicators and are not filtered
# -----------------------------
# Film-side continuous variables
film_continuous = [
    "film_vote_count", "film_popularity", "film_budget", "film_revenue",
    "film_rating", "days_since_film_release"
]

# Film-side binary variables all start with "film_is"
film_binary = [c for c in film_features if c.startswith("film_is_")]

# Album-side continuous and binary variables
album_continuous = ["days_since_album_release", "n_tracks", "composer_album_count"]
album_binary = [c for c in album_features if c not in album_continuous]

# Awards are binary flags (0/1)
awards_binary = derived_award_cols

continuous_features = film_continuous + album_continuous
binary_features = film_binary + album_binary + awards_binary


In [25]:
# -----------------------------
# Step 2) Light winnowing (continuous only)
# -----------------------------
df = album_analytics_df[binary_features + continuous_features + [target]].copy()

# Pearson correlations for continuous features vs target (pairwise complete)
cont_corr = df[[target] + continuous_features].corr(method="pearson")[target].drop(target)

kept_continuous = cont_corr[cont_corr.abs() >= THRESH].index.tolist()
dropped_continuous = cont_corr[cont_corr.abs() < THRESH].index.tolist()

# Final X list for OLS (binary all kept + filtered continuous)
# Final predictor list:
# - All binary flags
# - Only continuous predictors with a minimal linear signal vs target
X_cols = binary_features + kept_continuous

print(f"Continuous kept (|r| >= {THRESH:.2f}): {len(kept_continuous)} / {len(continuous_features)}")
print(f"Binary kept (no filter): {len(binary_features)}")
print(f"Final X feature count: {len(X_cols)}")

print("\nKept continuous features (sorted by |corr|):")
display(cont_corr.loc[kept_continuous].sort_values(key=lambda s: s.abs(), ascending=False).to_frame("pearson_r").round(3))

print("\nDropped continuous features:")
print(dropped_continuous)


Continuous kept (|r| >= 0.05): 6 / 9
Binary kept (no filter): 29
Final X feature count: 35

Kept continuous features (sorted by |corr|):


,pearson_r
film_vote_count,0.223
film_revenue,0.182
film_budget,0.158
film_rating,0.146
days_since_film_release,0.144
days_since_album_release,0.119



Dropped continuous features:
['film_popularity', 'n_tracks', 'composer_album_count']


### IV\.2 Transform features

Before fitting the regression, we pause to validate a core assumption of OLS: that continuous predictors relate to the outcome in an approximately linear, well\-behaved way\. Several of our film\-side variables \(votes, budget, revenue, popularity\) are known to be heavy\-tailed, and earlier correlation work already hinted that a small number of blockbuster films exert outsized influence\. Rather than transforming by default, we run a compact diagnostic: a faceted scatterplot of log\(album listeners\) against log1p\(feature\) for each continuous predictor\. This lets us quickly assess, in one view, whether log compression meaningfully improves linearity and variance stability, and whether any variables are better left on their original scale\. The goal here is pragmatic: apply only the transformations that materially improve model behavior, and leave the rest untouched\.

In [26]:
# This is my evaluation of each pair-chart. Placing it here so it can be printed right above its
# corresponding pairwise charts
eval_map = {
    "film_vote_count": "Heavy-tailed; log1p should reduce leverage from blockbusters and make the slope easier to read.",
    "film_budget": "Heavy-tailed; raw scale is dominated by a few extremes — log1p should stabilize the trend.",
    "film_revenue": "Extremely heavy-tailed; log1p should compress the top tail and reveal mid-range structure.",
    "film_popularity": "Skewed; log1p often spreads the low-end pile-up and clarifies the relationship.",
    "film_rating": "Bounded; log1p usually won’t change much — expect similar shape either way.",
    "days_since_film_release": "Timing variable; already smooth/linear-ish — log1p is typically unnecessary.",
    "days_since_album_release": "Timing variable; raw should already be interpretable — log1p rarely adds value.",
    "n_tracks": "Discrete/bounded; log1p won’t materially change the relationship (look for a weak/flat slope).",
    "composer_album_count": "Discrete; expect limited effect size \n log1p mainly just re-scales the x-axis.",
}


In [27]:
# ------------------------------------------------------------
# Step 3 Diagnostic (revised):
# Raw vs log1p(feature), with a single evaluation line BETWEEN the pair.
#
# Key idea:
# - We first build a "long" dataframe that contains one row per observation
#   per (feature, scale).
# - IMPORTANT: we filter OUT x==0 for each feature when building "long"
#   so vertical stacks at 0 don't appear in the charts.
# - Then we build the visualization as a vertical stack (vconcat):
#     [evaluation text line]
#     [two-column chart: raw vs log1p]
#   repeated for each feature.
# ------------------------------------------------------------

y = y_feature[0]  # e.g., 'log_lfm_album_listeners'

# ------------------------------------------------------------
# 1) Start from the analytics dataframe and keep only needed columns.
#    We only drop rows where y is missing (because the scatter y-axis needs it).
#    NOTE: This does NOT remove x==0; we handle that per-feature below.
# ------------------------------------------------------------
diag_df = album_analytics_df[continuous_features + [y]].dropna(subset=[y]).copy()

# ------------------------------------------------------------
# 2) Build a LONG dataframe with BOTH:
#    - raw x values
#    - log1p(x) values
#
#    We do this per feature so we can apply per-feature filtering rules:
#    - drop rows where the feature is missing
#    - drop rows where the feature is exactly 0 (your request)
#
#    Resulting schema of `long`:
#      [ y, x, feature, scale ]
# ------------------------------------------------------------
rows = []

for c in continuous_features:

    # --- Base subset for this feature only ---
    # Keep rows where:
    # - the feature value exists (not NaN)
    # - the feature value is NOT 0 (so we remove those vertical stacks)
    #
    # If you ever decide you want "drop <= 0" instead, change (d[c] != 0) to (d[c] > 0).
    base = (
        diag_df[[y, c]]
        .dropna()                       # drop rows where this feature is missing
        .loc[lambda d: d[c] != 0]       # drop rows where this feature equals 0
        .copy()
    )

    # --- Raw version: x = feature value as-is ---
    raw = (
        base.rename(columns={c: "x"})   # rename feature column to generic "x"
            .assign(feature=c, scale="raw")
    )

    # --- Log1p version: x = log(1 + feature) ---
    # log1p is stable for large values and avoids log(0) issues.
    # We keep .clip(lower=0) just in case negatives exist (should be rare here).
    log = (
        base.assign(x=lambda d: np.log1p(d[c].clip(lower=0)))
            .loc[:, [y, "x"]]           # keep only y and the new x
            .assign(feature=c, scale="log1p")
    )

    rows.append(raw)
    rows.append(log)

# Combine all features/scales into one long dataframe
long = pd.concat(rows, ignore_index=True)

# Make sure the facet columns appear in the intended order: raw then log1p
long["scale"] = pd.Categorical(long["scale"], categories=["raw", "log1p"], ordered=True)

# ------------------------------------------------------------
# 3) Build the visualization as a set of "blocks" (one block per feature):
#    Block layout:
#      (a) evaluation text row
#      (b) a paired chart: raw vs log1p (two columns)
#
#    Then we vertically stack (vconcat) all blocks.
# ------------------------------------------------------------

blocks = []

for c in continuous_features:
    # Get the human-readable evaluation note for this feature
    note = eval_map.get(c, "")

    # --- (a) Evaluation text row ---
    # This is a tiny one-row dataframe, so it renders ONCE (not duplicated).
    eval_row = (
        alt.Chart(pd.DataFrame({"text": [f"{c}: {note}"]}))
        .mark_text(
            align="left",
            baseline="middle",
            fontSize=12,
            dx=6
        )
        .encode(text="text:N")
        .properties(height=24)
    )

    # --- (b) Paired chart for this feature only ---
    # We subset `long` to the current feature, which already has x==0 removed.
    sub = long[long["feature"] == c]

    # Scatter points
    points = (
        alt.Chart(sub)
        .mark_circle(opacity=0.20, size=28)
        .encode(
            x=alt.X("x:Q", title=None),
            y=alt.Y(f"{y}:Q", title="log(album listeners)"),
        )
    )

    # Trendline: regression within each scale (raw vs log1p)
    trend = (
        alt.Chart(sub)
        .transform_regression("x", y, groupby=["scale"])
        .mark_line(strokeWidth=3, opacity=0.9)
        .encode(
            x="x:Q",
            y=f"{y}:Q"
        )
    )

    # Facet into two columns: raw (left) and log1p (right)
    pair = (points + trend).facet(
        column=alt.Column("scale:N", title=None, sort=["raw", "log1p"])
    ).resolve_scale(
        x="independent"   # raw and log1p have different x ranges, so don't force them to match
    )

    # Stack evaluation row ABOVE the paired chart for this feature
    blocks.append(alt.vconcat(eval_row, pair, spacing=6))

# Stack all feature-blocks vertically
chart = alt.vconcat(*blocks, spacing=14).resolve_scale(
    y="shared"  # keep y-axis consistent across all features for easier comparison
).properties(
    title={
        "text": "Raw vs log1p diagnostics for continuous predictors",
        "subtitle": [
            "Each feature is shown as a paired view (raw vs log1p).",
            "Rows where the feature value equals 0 are excluded from that feature’s plots."
        ]
    }
)

chart


alt.VConcatChart(...)

Findings: Looking across the nine diagnostic plots, a few patterns stood out pretty clearly\. Several film\-level exposure variables—especially revenue, vote count, and budget \(and to a lesser extent popularity\)—are very right\-skewed\. In the raw plots, the regression lines are strongly influenced by a small number of blockbuster films, while most albums are crowded into a much narrower range\. On the raw scale, this makes it hard to see how these variables relate to album listeners for the majority of observations\.

When these same variables are plotted using a log \(or log1p\) transform, the extreme values are pulled in and the overall patterns are easier to read\. The relationships don’t suddenly become strong or perfectly linear, but the fitted lines appear less dominated by outliers and more reflective of the bulk of the data\. Based on these diagnostics, using a log transform for these heavily skewed exposure variables seems reasonable before fitting linear models\.

In contrast, the timing variables \(days since film release and days since album release\) already show fairly smooth, straight\-forward relationships with log\(album listeners\) on the raw scale, and the log transform does not appear to change their behavior in a meaningful way\. Similarly, discrete and bounded features such as track count, composer album count, and film rating show weak but consistent patterns across both scales\. In these cases, the small slopes appear to reflect genuinely limited effects rather than problems with scaling\.

Overall, these diagnostics suggest that log\-transforming the most skewed exposure variables is helpful, while leaving timing and bounded variables on their original scales is likely sufficient for the regression models used later in the analysis\.

In [28]:

# -----------------------------
# Step 3 — Apply transforms
# -----------------------------
df_model = album_analytics_df[X_cols + [target]].copy()

# Identify predictor types
X_cont = [c for c in X_cols if c not in binary_features]  # continuous numeric predictors
X_bin  = [c for c in X_cols if c in binary_features]      # 0/1 flags (genres, awards, etc.)

# (a) Log-transform ONLY the heavy-tailed exposure variables (if present)
# SPLOM justification: these are strongly right-skewed and benefit from log compression.
HEAVY_TAILED = ["film_vote_count", "film_budget", "film_revenue", "film_popularity"]

for c in HEAVY_TAILED:
    if c in X_cont:
        # log1p handles zeros; clip avoids issues if anything weird slipped in
        df_model[c] = np.log1p(df_model[c].clip(lower=0))

# (b) Z-score all continuous predictors (post-log where applicable)
# Recommended because it puts continuous predictors on a common scale and makes
# coefficients easier to compare (does NOT change model fit).
for c in X_cont:
    mu = df_model[c].mean(skipna=True)
    sd = df_model[c].std(skipna=True, ddof=0)

    # Avoid divide-by-zero if a column is constant
    if sd == 0 or np.isnan(sd):
        df_model[c] = df_model[c] - mu
    else:
        df_model[c] = (df_model[c] - mu) / sd

# Binary predictors stay as 0/1 (no scaling, no transforms)
# Target is already log-transformed upstream in your pipeline.

print("Step 3 complete.")
print(f"Modeling frame shape: {df_model.shape}")
print(f"Continuous predictors standardized: {len(X_cont)}")
print(f"Binary predictors untouched: {len(X_bin)}")
print("Logged predictors (if present):", [c for c in HEAVY_TAILED if c in X_cont])

# df_model is now ready for Step 4 (OLS)


Step 3 complete.
Modeling frame shape: (1551, 36)
Continuous predictors standardized: 6
Binary predictors untouched: 29
Logged predictors (if present): ['film_vote_count', 'film_budget', 'film_revenue']


Based on the diagnostic plots, I log\-transformed only the film exposure variables that showed strong right skew \(vote count, budget, and revenue\)\. All continuous predictors were then standardized so they are on a common scale, while binary indicators were left unchanged\. The target variable was already log\-transformed earlier\. This produces a clean modeling dataset that is ready for OLS regression\.

### IV\.3 Simple linear regression

Let's do a final check and cleanup of our df\_model before running regression\. We should drop addition features that we've seen to be highly correlated to reduce multicollinearity before running regression

In [29]:
print(df_model.columns)

# To prevent multicollinearity:
# 1) Drop days_since_film_release (nearly perfectly correlated with days_since_album_release)
# 2) Keep only ONE film exposure proxy (film_vote_count) and drop the rest
df_model_final = df_model.drop(columns=[
    "days_since_film_release",
    "film_budget",
    "film_revenue"
])

# The film_is columns are boolean -- convert them to int
film_genre_cols = [c for c in df_model_final.columns if c.startswith("film_is_")]

df_model_final[film_genre_cols] = df_model_final[film_genre_cols].astype(int)

print("Final model:", df_model_final.columns)

Index(['film_is_action', 'film_is_adventure', 'film_is_animation', 'film_is_comedy', 'film_is_crime', 'film_is_documentary', 'film_is_drama', 'film_is_family', 'film_is_fantasy', 'film_is_history',
       'film_is_horror', 'film_is_music', 'film_is_mystery', 'film_is_romance', 'film_is_science_fiction', 'film_is_tv_movie', 'film_is_thriller', 'film_is_war', 'film_is_western',
       'ambient_experimental', 'classical_orchestral', 'electronic', 'hip_hop_rnb', 'pop', 'rock', 'world_folk', 'us_score_nominee_count', 'us_song_nominee_count', 'bafta_nominee', 'film_vote_count',
       'film_budget', 'film_revenue', 'film_rating', 'days_since_film_release', 'days_since_album_release', 'log_lfm_album_listeners'],
      dtype='object')
Final model: Index(['film_is_action', 'film_is_adventure', 'film_is_animation', 'film_is_comedy', 'film_is_crime', 'film_is_documentary', 'film_is_drama', 'film_is_family', 'film_is_fantasy', 'film_is_history',
       'film_is_horror', 'film_is_music', 'film_is_m

At this stage, df\_model already contains only the finalized predictors, so the regression uses all remaining columns except the target variable\.

In [30]:
# ------------------------------------------------------------
# Final OLS regression (minimal + correct ordering)
# ------------------------------------------------------------

y_col = "log_lfm_album_listeners"

# Use all remaining columns except the target
X_cols = [c for c in df_model_final.columns if c != y_col]

# ------------------------------------------------------------
# 1) Build modeling frame and force numeric types
# ------------------------------------------------------------

model_df_reg = df_model_final[[y_col] + X_cols].copy()

# Coerce everything to numeric (required by statsmodels)
model_df_reg = model_df_reg.apply(pd.to_numeric, errors="coerce")

# Drop rows with any missing values
model_df_reg = model_df_reg.dropna()

print("Modeling rows:", model_df_reg.shape[0])
print("Predictors:", len(X_cols))

# ------------------------------------------------------------
# 2) Split into X and y
# ------------------------------------------------------------

y = model_df_reg[y_col]
X = model_df_reg[X_cols]

# Add intercept
X = sm.add_constant(X)

# ------------------------------------------------------------
# 3) Fit OLS model
# ------------------------------------------------------------

results = sm.OLS(y, X).fit()

print(results.summary())


Modeling rows: 744
Predictors: 32
                               OLS Regression Results                              
Dep. Variable:     log_lfm_album_listeners   R-squared:                       0.141
Model:                                 OLS   Adj. R-squared:                  0.102
Method:                      Least Squares   F-statistic:                     3.645
Date:                     Fri, 13 Feb 2026   Prob (F-statistic):           1.58e-10
Time:                             19:58:28   Log-Likelihood:                -1719.0
No. Observations:                      744   AIC:                             3504.
Df Residuals:                          711   BIC:                             3656.
Df Model:                               32                                         
Covariance Type:                 nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------

Findings\. Using a simplified OLS model with 744 soundtrack albums and 32 predictors, I find that overall film exposure is the strongest and most consistent driver of soundtrack popularity\. Film vote count shows a large, positive, and highly significant association with log album listeners, even after controlling for film genres, album genres, timing, and awards\. This reinforces the idea that soundtrack listening is closely tied to how widely seen or discussed the underlying film is, rather than to many of the finer\-grained film or album characteristics\.

Several content\-related signals emerge in intuitive ways\. Animated films and documentaries are associated with substantially higher soundtrack listening, and music\-focused films also show a positive effect that is close to conventional significance thresholds\. On the album side, classical/orchestral soundtracks are positively associated with listener counts, while electronic soundtracks show a similar but slightly weaker pattern\. These effects suggest that certain film types and musical styles are more likely to generate sustained soundtrack engagement\.

In contrast, most genre indicators—both film and album—do not show strong independent effects once exposure is accounted for\. This appears to reflect the fact that genre primarily shapes what kind of soundtrack a listener encounters, rather than how many listeners it ultimately attracts\. Similarly, award nominations \(both U\.S\. and BAFTA\) do not show a meaningful relationship with soundtrack listening in this model, suggesting that critical recognition does not translate directly into broader listener demand once exposure is controlled\.

Finally, timing effects are modest and statistically insignificant, indicating that once a soundtrack has been released, simple “age since release” does not strongly explain differences in cumulative listener counts\. Overall, the model explains a meaningful but limited share of variation in soundtrack popularity \(R² ≈ 0\.14\), which is reasonable given the many unobserved factors that influence music consumption\. Taken together, these results support a simple conclusion: soundtrack popularity is driven primarily by film reach, with secondary contributions from certain film categories and musical styles, rather than by awards or fine\-grained genre distinctions\.

### IV\.4 Regression Visualizations

Our strongest variable is film\_vote\_count \-\- so let's generate a scatterplot of it against log album listeners

In [31]:
# Use the exact regression sample
plot_df = model_df_reg[["film_vote_count", "log_lfm_album_listeners"]].copy()

# Fit simple line in Python: y = a*x + b
x = plot_df["film_vote_count"].to_numpy()
y = plot_df["log_lfm_album_listeners"].to_numpy()
a, b = np.polyfit(x, y, 1)   # Fit the best-fitting line across the datapoints

# Make a 2-point line spanning the x-range
x_min, x_max = float(x.min()), float(x.max())
line_df = pd.DataFrame({
    "film_vote_count": [x_min, x_max],
    "log_lfm_album_listeners": [a * x_min + b, a * x_max + b]
})

scatter = alt.Chart(plot_df).mark_circle(opacity=0.25, size=40).encode(
    x=alt.X("film_vote_count:Q", title="Film vote count (exposure proxy)"),
    y=alt.Y("log_lfm_album_listeners:Q", title="Log soundtrack listeners")
)

line = alt.Chart(line_df).mark_line(strokeWidth=3).encode(
    x="film_vote_count:Q",
    y="log_lfm_album_listeners:Q"
)

(scatter + line).properties(
    width=650,
    height=400,
    title="Film exposure vs soundtrack popularity (with fitted line)"
)


alt.LayerChart(...)

Findings: This figure shows the relationship between film exposure and soundtrack popularity for the albums used in the regression\. Each point is a soundtrack album, with film exposure measured by film vote count and popularity measured as log soundtrack listeners\. Film vote count was log\-transformed earlier in the analysis, which is why the x\-axis values look compressed and include negative numbers\.

Despite substantial scatter, there is a clear positive trend: soundtracks tied to more widely voted\-on films tend to attract more listeners on average\. The fitted line reflects this pattern and mirrors the regression result, where film vote count emerged as the strongest and most consistent predictor of soundtrack popularity\.

In [32]:
# -----------------------------
# 1) Build tidy coefficient table
# -----------------------------
coef_df = (
    pd.DataFrame({
        "feature": results.params.index,
        "coef": results.params.values,
        "ci_low": results.conf_int()[0].values,
        "ci_high": results.conf_int()[1].values,
    })
    .query("feature != 'const'")
    .copy()
)

# CI crosses zero? (True/False)
coef_df["crosses_zero"] = (coef_df["ci_low"] <= 0) & (coef_df["ci_high"] >= 0)

# Make it a *string category* so Altair treats it as two discrete groups reliably
coef_df["ci_group"] = coef_df["crosses_zero"].map(
    {True: "CI crosses 0", False: "CI does NOT cross 0"}
)

# Sort so the largest absolute effects appear first (top)
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)

# Lock the y-order to this sorted order
y_order = coef_df["feature"].tolist()

In [33]:
# -----------------------------
# 2) Dot-and-whisker plot (theme colors)
# -----------------------------

whiskers = alt.Chart(coef_df).mark_rule().encode(
    x="ci_low:Q",
    x2="ci_high:Q",
    y=alt.Y("feature:N", sort=y_order, title=None),
    color=alt.Color("ci_group:N", title=None)  # uses theme category palette
)

dots = alt.Chart(coef_df).mark_circle(size=80).encode(
    x=alt.X("coef:Q", title="Effect on log soundtrack listeners"),
    y=alt.Y("feature:N", sort=y_order, title=None),
    color=alt.Color("ci_group:N", title=None, legend=None),
    tooltip=[
        alt.Tooltip("feature:N", title="Feature"),
        alt.Tooltip("coef:Q", format=".3f", title="Coefficient"),
        alt.Tooltip("ci_low:Q", format=".3f", title="CI low"),
        alt.Tooltip("ci_high:Q", format=".3f", title="CI high"),
        alt.Tooltip("ci_group:N", title="")
    ]
)

zero = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(strokeDash=[4, 4]).encode(
    x="x:Q"
)

chart = (whiskers + dots + zero).properties(
    width=750,
    height=700,
    title={
        "text": "Regression coefficients with 95% confidence intervals",
        "subtitle": [
            "Dots are coefficient estimates; whiskers are 95% confidence intervals.",
            "Color indicates whether the confidence interval crosses 0."
        ]
    }
)

chart

alt.LayerChart(...)

Findings: This chart shows the regression coefficients with their 95% confidence intervals\. Each dot is the estimated effect of a feature on log soundtrack listeners, and the horizontal lines show how uncertain that estimate is\. The dashed vertical line marks zero \(no effect\)\.

A small number of variables clearly stand out because their confidence intervals do not cross zero — most notably film vote count \(our exposure proxy\), documentary films, animation, and a few music\-related genres\. These are the features where the model suggests a more reliable association with soundtrack popularity\.

Most other features have confidence intervals that cross zero, which means their estimated effects are noisy and could plausibly be positive or negative\. I interpret those as weak or inconclusive signals rather than meaningful drivers\. Overall, this reinforces the idea that film exposure matters most, while genre and award features tend to have smaller and less stable effects\.

In [34]:
# We should export the album_analytics_df (before the extreme feature pruning we did) for the final visualization
display(album_analytics_df.head())

album_analytics_df.to_csv("./pipeline/5.5.Albums_for_final_viz.csv", index = False)

,film_vote_count,film_popularity,film_budget,film_revenue,film_rating,days_since_film_release,film_runtime_min,film_is_action,film_is_adventure,film_is_animation,film_is_comedy,film_is_crime,film_is_documentary,film_is_drama,film_is_family,film_is_fantasy,film_is_history,film_is_horror,film_is_music,film_is_mystery,film_is_romance,film_is_science_fiction,film_is_tv_movie,film_is_thriller,film_is_war,film_is_western,days_since_album_release,n_tracks,composer_album_count,ambient_experimental,classical_orchestral,electronic,hip_hop_rnb,pop,rock,world_folk,us_score_nominee_count,us_song_nominee_count,bafta_nominee,log_lfm_album_listeners
0,12222,18.4171,40000000,569651467,5.875,4007,125,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,True,False,False,4001.0,16.0,19,0,0,0,0,0,0,0,0,3,0,3.295837
1,7294,11.8041,95000000,543514353,6.826,3984,105,False,False,False,False,False,False,True,True,True,False,False,False,False,True,False,False,False,False,False,3980.0,33.0,6,0,1,0,0,0,0,0,0,0,0,6.261492
2,18928,10.1655,135000000,532950503,7.540,3690,157,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,3690.0,22.0,2,1,1,0,0,0,0,0,2,0,1,2.639057
3,20206,10.1330,245000000,2068223624,7.300,3700,136,True,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,4048.0,23.0,6,0,1,0,0,0,0,0,1,0,1,5.780744
4,9272,9.7665,30000000,84997446,7.415,3789,122,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,3788.0,18.0,3,0,1,1,0,0,0,0,2,0,0,6.725034


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b124131d-2024-4253-bb46-8043aed4b78f' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>